In [1]:
import pandas as pd

# Load dataset
data = pd.read_csv("KDDTrain+.txt", header=None)

print("Original Shape:", data.shape)


Original Shape: (125973, 43)


In [2]:
# Column -2 is label
labels = data.iloc[:, -2]

# Drop last two columns (label + difficulty)
features = data.iloc[:, :-2]

print("Features shape:", features.shape)
print("Labels unique:", labels.unique()[:10])


Features shape: (125973, 41)
Labels unique: ['normal' 'neptune' 'warezclient' 'ipsweep' 'portsweep' 'teardrop' 'nmap'
 'satan' 'smurf' 'pod']


In [3]:
labels = labels.apply(lambda x: 0 if x == "normal" else 1)

print(labels.value_counts())


41
0    67343
1    58630
Name: count, dtype: int64


In [4]:
data_clean = features.copy()
data_clean["label"] = labels

print(data_clean.shape)


(125973, 42)


In [5]:
print(labels.value_counts())


41
0    67343
1    58630
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X = features
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:")
print(y_train.value_counts())


Train distribution:
41
0    53874
1    46904
Name: count, dtype: int64


In [7]:
# One-hot encoding
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# Align columns (important!)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)


In [8]:
print(X_train.select_dtypes(include=['object']).columns)



Index([], dtype='object')


In [9]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)


In [10]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)


X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

# 3️⃣ Convert explicitly to numeric numpy arrays
X_train = X_train.values
X_test = X_test.values

# 4️⃣ Now scale
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [11]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [12]:
print("New shape:", X_train.shape)


New shape: (100778, 121, 1)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

model = Sequential()
model.add(Input(shape=(X_train.shape[1], 1)))


model.add(LSTM(64, return_sequences=True))
model.add(Dropout(0.3))


model.add(LSTM(32))
model.add(Dropout(0.3))


model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 121, 64)             │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 121, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           1,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 30,401 (118.75 KB)

 Trainable params: 30,401 (118.75 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [15]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)


Epoch 1/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 84s 65ms/step - accuracy: 0.9271 - loss: 0.2216 - val_accuracy: 0.9553 - val_loss: 0.1416
Epoch 2/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 83s 66ms/step - accuracy: 0.9592 - loss: 0.1404 - val_accuracy: 0.9613 - val_loss: 0.1299
Epoch 3/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 80s 63ms/step - accuracy: 0.9626 - loss: 0.1278 - val_accuracy: 0.9592 - val_loss: 0.1296
Epoch 4/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 83s 66ms/step - accuracy: 0.9636 - loss: 0.1221 - val_accuracy: 0.9671 - val_loss: 0.1160
Epoch 5/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 81s 65ms/step - accuracy: 0.9647 - loss: 0.1188 - val_accuracy: 0.9574 - val_loss: 0.1305
Epoch 6/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 81s 64ms/step - accuracy: 0.9655 - loss: 0.1146 - val_accuracy: 0.9658 - val_loss: 0.1073
Epoch 7/30
1260/1260 ━━━━━━━━━━━━━━━━━━━━ 81s 64ms/step - accuracy: 0.9589 - loss: 0.1319 - val_accuracy: 0.9633 - val_loss: 0.1213
Epoch 8/30
 182/1260 ━━━━━━━━━━━━━━━━━━━━ 1:05 60ms/step - accuracy: 0.9638 

KeyboardInterrupt: 

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get predicted probabilities
y_pred_prob = model.predict(X_test)


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

roc_auc = auc(fpr, tpr)

print("AUC Score:", roc_auc)


In [ ]:
plt.figure()
plt.plot(fpr, tpr, label="LSTM (AUC = %0.4f)" % roc_auc)
plt.plot([0, 1], [0, 1], linestyle='--')  # Random line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()


In [ ]:
!pip install pandas numpy scikit-learn tensorflow

In [ ]:
import sys
!{sys.executable} -m pip install pandas

In [ ]:
import pandas as pd
print("Pandas working ✅")

In [ ]:
!pip install scikit-learn

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow keras

In [ ]:
!pip install tensorflow

In [ ]:
!python -m pip install --upgrade pip


In [ ]:
from tensorflow.keras.models import Sequential


In [ ]:
import numpy as np

In [ ]:
sample = X_test[0]   # ek real data point
sample = np.reshape(sample, (1, sample.shape[0], 1))

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

print(classification_report(y_test, y_pred))

In [16]:
import numpy as np

sample = X_test[10]   # koi bhi index le lo
sample = np.reshape(sample, (1, sample.shape[0], 1))

prediction = model.predict(sample)

print("Prediction:", prediction)
print("Actual:", y_test[10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 666ms/step
Prediction: [[0.01320807]]


KeyError: 10

In [17]:
import numpy as np

sample = X_test[10]
sample = np.reshape(sample, (1, sample.shape[0], 1))

prediction = model.predict(sample)

print("Prediction:", prediction)
print("Actual:", y_test.iloc[10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Prediction: [[0.01320807]]
Actual: 0


In [18]:
model.save("ids_model.h5")

In [19]:
model.save("ids_model.keras")